In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
!pip install mlflow xgboost
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import mlflow
import mlflow.sklearn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.4/49.4 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 78.4 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 77.2 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 46.4 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.3/86.3 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 907.5/907.5 kB 42.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.9/214.9 kB 11.4 MB/s eta 0:00:00


Data Generation

In [3]:
np.random.seed(42)
n_samples = 10000

temperature = np.random.normal(75, 15, n_samples)
vibration = np.random.normal(0.5, 0.2, n_samples)
pressure = np.random.normal(100, 20, n_samples)
rpm = np.random.normal(1500, 200, n_samples)
age_days = np.random.randint(0, 365, n_samples)

failure_score = (temperature > 90) * 0.3 + (vibration > 0.8) * 0.3 + (pressure > 130) * 0.2 + (age_days > 300) * 0.2
failure_prob = failure_score + np.random.normal(0, 0.1, n_samples)
failure = (failure_prob > 0.5).astype(int)

data = pd.DataFrame({'temperature': temperature, 'vibration': vibration, 'pressure': pressure, 'rpm': rpm, 'age_days': age_days, 'failure': failure})
print(f'Dataset shape: {data.shape}')
print(f'Failure rate: {data.failure.mean():.2%}')

Dataset shape: (10000, 6)
Failure rate: 4.15%


Preprocessing

In [4]:
X = data.drop('failure', axis=1)
y = data['failure']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

MLflow Setup

In [5]:
mlflow.set_experiment('predictive-maintenance')

2026/06/06 16:44:44 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/06/06 16:44:44 INFO mlflow.store.db.utils: Updating database tables
2026/06/06 16:44:46 INFO mlflow.tracking.fluent: Experiment with name 'predictive-maintenance' does not exist. Creating a new experiment.


<Experiment: artifact_location='/kaggle/working/mlruns/1', creation_time=1780764286954, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1780764286954, lifecycle_stage='active', name='predictive-maintenance', tags={}, trace_location=None, workspace='default'>

Models Training & Logging

A. Logistic Regression:


In [6]:
with mlflow.start_run(run_name='logistic_regression'):
    model = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]
    
    mlflow.log_param('model_type', 'LogisticRegression')
    mlflow.log_metric('accuracy', accuracy_score(y_test, y_pred))
    mlflow.log_metric('f1_score', f1_score(y_test, y_pred))
    mlflow.sklearn.log_model(model, 'model')
    print("Logistic Regression trained.")

2026/06/06 16:44:47 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/06 16:44:47 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Logistic Regression trained.


Random Forest:

In [7]:
with mlflow.start_run(run_name='random_forest'):
    model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    
    mlflow.log_metric('accuracy', accuracy_score(y_test, y_pred))
    mlflow.log_metric('f1_score', f1_score(y_test, y_pred))
    mlflow.sklearn.log_model(model, 'model')
    print("Random Forest trained.")

2026/06/06 16:45:01 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/06 16:45:01 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Random Forest trained.


C. XGBoost:

In [8]:
with mlflow.start_run(run_name='xgboost'):
    model = XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42)
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    
    mlflow.log_metric('accuracy', accuracy_score(y_test, y_pred))
    mlflow.log_metric('f1_score', f1_score(y_test, y_pred))
    mlflow.sklearn.log_model(model, 'model')
    print("XGBoost trained.")

2026/06/06 16:45:06 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/06 16:45:06 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


XGBoost trained.


Lab 13: Teaching the Computer (Training)

In this lab, I focused on building and testing your machine learning model.

    Preparing Data: I organized sensor data (like heat, vibration, and pressure).

    Training: I used the XGBoost algorithm to teach the computer how to predict when a machine would break.

    Tracking: I used a tool called MLflow to save every detail of your experiments—like how accurate the model was and what settings (hyperparameters) you used.

    Saving: I saved the finished model so it could be used later.

LAB 14 

In [10]:
import mlflow
from mlflow.tracking import MlflowClient
import pandas as pd

# Database path set karein
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment('predictive-maintenance')

client = MlflowClient()
print("Connected to existing database!")

Connected to existing database!


In [11]:
experiment = client.get_experiment_by_name('predictive-maintenance')
runs = client.search_runs(experiment_ids=[experiment.experiment_id])

print(f"Total runs found: {len(runs)}")

# Agar runs mil gaye, toh ek table display karein
if len(runs) > 0:
    run_data = [{'run_id': r.info.run_id[:8], 'roc_auc': r.data.metrics.get('roc_auc', 0)} for r in runs]
    print(pd.DataFrame(run_data))

Total runs found: 3
     run_id  roc_auc
0  978805e3        0
1  5fcd4a45        0
2  624ae554        0


In [13]:
# Yeh code automatically sabse best (highest roc_auc) run ki ID nikal dega
best_run = client.search_runs(
    experiment_ids=['1'], 
    order_by=['metrics.roc_auc DESC']
)[0] # Pehla run jo best hai

best_run_id = best_run.info.run_id
print(f"Best Run ID: {best_run_id}")

# Ab register karein
model_uri = f"runs:/{best_run_id}/model"
model_name = "PredictiveMaintenanceModel"

mlflow.register_model(model_uri=model_uri, name=model_name)
print(f"Model {model_name} successfully registered!")

Registered model 'PredictiveMaintenanceModel' already exists. Creating a new version of this model...
2026/06/06 17:06:14 WARNING mlflow.tracking._model_registry.fluent: Run with id 978805e376d2433a9abc97133156fb1a has no artifacts at artifact path 'model', registering model based on models:/m-8272089e083542fd81c6a112dd81209b instead


Best Run ID: 978805e376d2433a9abc97133156fb1a
Model PredictiveMaintenanceModel successfully registered!


Created version '1' of model 'PredictiveMaintenanceModel'.


In [14]:
# Model ko Staging mein transition karein
client.transition_model_version_stage(
    name="PredictiveMaintenanceModel",
    version="1",
    stage="Staging"
)
print("Model moved to Staging successfully!")

Model moved to Staging successfully!


/tmp/ipykernel_58/730897243.py:2: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


In [15]:
# Model ko Staging se load karein
model_uri = "models:/PredictiveMaintenanceModel/Staging"
model = mlflow.pyfunc.load_model(model_uri)

# Inference function
def predict_equipment_failure(input_data):
    """
    input_data: DataFrame jo naya sensor data contain karta hai
    """
    predictions = model.predict(input_data)
    return predictions

print("Inference pipeline is ready to use!")

Inference pipeline is ready to use!


In [16]:
# Model ko Production mein transition karein
client.transition_model_version_stage(
    name="PredictiveMaintenanceModel",
    version="1",
    stage="Production"
)
print("Model successfully promoted to Production!")

Model successfully promoted to Production!


/tmp/ipykernel_58/4270948489.py:2: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


In [20]:
# Agar X_train variable memory mein hai
sample_input = X_train.iloc[[0]] 
print("Sample Input columns:", sample_input.columns.tolist())

# Prediction
prediction = production_model.predict(sample_input)
print("Prediction successful:", prediction)

Sample Input columns: ['temperature', 'vibration', 'pressure', 'rpm', 'age_days']
Prediction successful: [1]


In [21]:
def predict_equipment_failure(temperature, vibration, pressure, rpm, age_days):
    # Data ko DataFrame mein convert karna
    input_data = pd.DataFrame({
        'temperature': [temperature],
        'vibration': [vibration],
        'pressure': [pressure],
        'rpm': [rpm],
        'age_days': [age_days]
    })
    
    # Model se prediction lena
    prediction = production_model.predict(input_data)
    
    # Human-readable format mein return karna
    result = "Failure Predicted" if prediction[0] == 1 else "Normal Operation"
    return result

# Test run
print(predict_equipment_failure(75.5, 0.02, 1013, 3000, 150))

Failure Predicted


In [24]:
# 1. Model Description aur Tags add karein
client.update_registered_model(
    name="PredictiveMaintenanceModel", 
    description="Model for predictive maintenance using XGBoost."
)
client.set_registered_model_tag("PredictiveMaintenanceModel", "team", "DataScience")
client.set_registered_model_tag("PredictiveMaintenanceModel", "framework", "xgboost")
client.set_registered_model_tag("PredictiveMaintenanceModel", "validation_status", "approved")

print("Metadata and tags updated successfully.")

Metadata and tags updated successfully.


In [25]:
# 3 Scenarios
print(predict_equipment_failure(75.5, 0.02, 1013, 3000, 150)) # Scenario 1
print(predict_equipment_failure(30.0, 0.01, 1000, 2000, 10))  # Scenario 2
print(predict_equipment_failure(90.0, 0.05, 1200, 4000, 500)) # Scenario 3

Failure Predicted
Failure Predicted
Failure Predicted


In [26]:
# Agar Version 2 mein issue ho, toh wapas Version 1 ko Production mein set karein
client.transition_model_version_stage(
    name="PredictiveMaintenanceModel",
    version="1",
    stage="Production"
)
print("Rollback to Version 1 successful.")

Rollback to Version 1 successful.


/tmp/ipykernel_58/832302632.py:2: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


In [27]:
# Naya run_id ya purana run_id use karte hue Version 2 register karein
model_uri = f"runs:/{best_run_id}/model"
mlflow.register_model(model_uri=model_uri, name="PredictiveMaintenanceModel")
print("Version 2 registered successfully.")

Registered model 'PredictiveMaintenanceModel' already exists. Creating a new version of this model...
2026/06/06 17:13:38 WARNING mlflow.tracking._model_registry.fluent: Run with id 978805e376d2433a9abc97133156fb1a has no artifacts at artifact path 'model', registering model based on models:/m-8272089e083542fd81c6a112dd81209b instead


Version 2 registered successfully.


Created version '2' of model 'PredictiveMaintenanceModel'.


Lab 14: Making the Model Ready for Use (Deployment)

In this lab, I took that saved model and turned it into a professional product.

    Registry:I officially "registered" my model in a library, giving it versions (like Version 1 and Version 2) so i can keep track of changes.

    Descriptions & Tags: I added notes (tags) to the model so that others know who built it and what it does.

    Moving to Production: I moved the model through stages. I tested it in "Staging" (the testing area) and then promoted it to "Production" (the final live version).

    Rollback Test: I learned how to quickly switch back to an older version (Version 1) if the new version causes problems.

    Prediction Function: I created a simple function called predict_equipment_failure(). Now, whenever I give it new data, it tells me immediately if the machine is going to fail or if it is working normally.

    Testing: I tested this function with different scenarios to make sure it gives the right answers.